# String and List Output Parsers [Step 4 - String and list parsers]

> **MLCourse - Agentic AI - LCEL and Runnables**

> Stage in the capstone: the entire RAG pipeline IS one runnable chain; memory
> wrapping uses RunnableWithMessageHistory.

### What you'll learn
- what a chat model actually returns (`AIMessage`) and why raw objects are awkward to consume
- `StrOutputParser`: the one-liner that extracts plain text, and how it pipes at the END of a chain
- `CommaSeparatedListOutputParser`: its format instructions shown verbatim, embedded correctly, parsed live and offline
- why parser POSITION in the pipe matters, plus a bridge to the Pydantic tier from Step 3

This is your first LCEL chain: template piped to model piped to parser. Every model
call below is guarded, and every parsing step has an offline twin.

### Standard first cell for every MLCourse notebook: imports, inline plotting,


In [ ]:
# and automatic discovery of the track-level .env file.
import os                                   # read environment variables such as GROQ_API_KEY
from pathlib import Path                    # walk up the folder tree hunting for .env

try:                                        # Jupyter kernels define get_ipython();
    get_ipython().run_line_magic("matplotlib", "inline")  # render plots inside the notebook
except NameError:                           # plain python runs have no IPython,
    pass                                    # so skip the magic silently

from dotenv import load_dotenv              # loads KEY=VALUE lines into os.environ


def find_track_env(start: Path) -> "Path | None":
    """Climb from *start* upward until 03_agentic_ai/.env appears."""
    for folder in (start, *start.parents):             # current dir, then every parent
        candidate = folder / "03_agentic_ai" / ".env"  # track secrets live at this spot
        if candidate.is_file():                        # hit: stop climbing immediately
            return candidate
    return None                                        # miss everywhere: caller decides


_env_path = find_track_env(Path.cwd())     # search from wherever the kernel started
if _env_path is not None:                  # found the track root?
    load_dotenv(_env_path)                 # push GROQ_API_KEY etc. into os.environ
    print("[setup] loaded env:", _env_path)
else:
    print("[setup] no 03_agentic_ai/.env found - live demos will be skipped")


### 1. From AIMessage to plain string

A chat model never returns a bare string. You get an `AIMessage` carrying content,
metadata, token usage, tool-call slots, and more. That richness is essential for
agents but annoying when you just want text to print, store, or feed onward.

`StrOutputParser` is the adapter: it pulls `.content` out of any message-shaped
input and hands you a `str`. Because it is itself a Runnable, it composes with `|`
at the END of a chain - the standard closing move of almost every pipeline.

> **Pro tip:** keep BOTH versions handy while debugging: the unparsed chain (to see
> metadata like token counts) and the parsed chain (for clean values). Switching is
> just appending or removing one `| StrOutputParser()`.

In [2]:
from langchain_core.messages import AIMessage               # what chat models return
from langchain_core.output_parsers import StrOutputParser   # AIMessage -> str
from langchain_core.prompts import ChatPromptTemplate       # dict -> rendered messages

if not os.getenv("GROQ_API_KEY"):           # mandatory guard before any provider call
    print("[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env to run this live call.")
else:
    from langchain_groq import ChatGroq     # official Groq integration package

    llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
    hello_prompt = ChatPromptTemplate.from_template(
        "Say hello to {name} in one short sentence."
    )

    raw_chain = hello_prompt | llm                    # two stages: stops AT the AIMessage
    raw = raw_chain.invoke({"name": "Ada"})
    print(type(raw).__name__, "| content repr:", repr(raw.content)[:70])

    clean_chain = hello_prompt | llm | StrOutputParser()   # parser LAST: message -> str
    clean = clean_chain.invoke({"name": "Ada"})
    print(type(clean).__name__, "| value:", repr(clean)[:70])

# Offline twin: the parser alone needs no provider whatsoever - it is pure code.
offline_msg = AIMessage(content="Hello, Ada! Lovely to meet you.")
print("offline parse:", repr(StrOutputParser().invoke(offline_msg)))

AIMessage | content repr: 'Hello, Ada!'


TextAccessor | value: 'Hello, Ada!'
offline parse: 'Hello, Ada! Lovely to meet you.'


### 2. Lists via CommaSeparatedListOutputParser

When you want a LIST of short items, asking for prose then splitting yourself is
fragile. LangChain ships a dedicated parser whose `.get_format_instructions()` is a
tiny contract you paste INTO the prompt - printed verbatim below. The model echoes
comma-separated items; the parser splits them into a real Python list.

> **Common pitfall:** building the list parser but forgetting to embed its format
> instructions. The parser only SPLITS text - it cannot force the model to produce
> comma-separated output. Without `{format}` in the template you get sentences, and
> the "list" becomes one giant mangled entry.

In [3]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

list_parser = CommaSeparatedListOutputParser()      # stateless: safe to reuse everywhere
instructions = list_parser.get_format_instructions()
print(instructions)                                  # THE exact contract text, verbatim

print(list_parser.parse("iron, calcium ,vitamin d")) # offline: deterministic split...
# ...note 'calcium ' keeps its trailing space! Normalize downstream when it matters:
print([item.strip().lower() for item in list_parser.parse("Iron, CALCIUM ,vitamin d")])

if not os.getenv("GROQ_API_KEY"):           # guard the live half of this section
    print("[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env to run this live call.")
else:
    from langchain_groq import ChatGroq

    llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
    vitamins_prompt = ChatPromptTemplate.from_template(
        "List 5 examples of {things}.\n{format}"        # instructions RIDE INSIDE the prompt
    ).partial(format=instructions)                      # injected at build time

    vitamin_chain = vitamins_prompt | llm | list_parser  # parser at the END again
    vitamins = vitamin_chain.invoke({"things": "essential vitamins"})
    print(type(vitamins).__name__, "->", vitamins)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`
['iron', 'calcium ', 'vitamin d']
['iron', 'calcium', 'vitamin d']


list -> ['Vitamin A', 'Vitamin B12', 'Vitamin C', 'Vitamin D', 'Vitamin E']


### 3. Position matters: the parser goes LAST

Read chains aloud left to right and track types as they flow:

- `prompt` consumes a dict, produces a prompt value (rendered messages).
- `llm` consumes messages, produces an `AIMessage`.
- `parser` consumes that message, produces your final Python type.

Each stage's INPUT type must equal its left neighbor's OUTPUT type. Put the parser
first and it receives a dict instead of a message - an immediate, confusing-looking
failure. The fix is mechanical: parsers terminate the chain.

> **Pro tip:** every parser is also runnable STANDALONE. `.invoke()` or `.parse()`
> a hand-written string any time you want to test the conversion without calling a
> model - which is exactly how we have been keeping sections above offline-friendly.

In [4]:
standalone = list_parser.invoke("mercury, venus, earth")   # same code path as inside chains
via_parse = list_parser.parse("mercury, venus, earth")     # .parse is the direct alias
print(standalone == via_parse, standalone)                 # True - identical behavior

True ['mercury', 'venus', 'earth']


### 4. Recap bridge to structured schemas

You now own the two cheapest tiers: strings and lists. The next tier up is JSON
(`JsonOutputParser` -> plain dicts), and beyond that sits Pydantic validation -
the Step 3 module's `AnswerWithSources` contract, where fields get TYPES and ranges
and bad fills raise instead of slipping through. Keep that ladder in mind whenever
you choose a parser: pick the strictest tier your use case can justify.

In [5]:
import json                                                  # for pretty-printing the dict
from langchain_core.output_parsers import JsonOutputParser   # tier three: text -> dict

jparse = JsonOutputParser()
as_dict = jparse.parse('{"status": "ok", "items": ["a", "b"]}')
print(type(as_dict).__name__, as_dict)      # flexible... but UNVALIDATED keys and types

dict {'status': 'ok', 'items': ['a', 'b']}


### Summary

- Chat models return `AIMessage` objects; `StrOutputParser` reduces them to plain
  strings and closes most chains.
- `CommaSeparatedListOutputParser` turns instructed text into `list[str]`; embed its
  format instructions in the prompt or the model never learns the contract.
- Parsers pipe at the END because chains are typed flows from dict to messages to data.
- Every parsing step demonstrated has an offline twin - parse locally, call models rarely.
- Next step on the ladder: validated schemas via Pydantic (Step 3 notebooks), then
  full pipeline composition later in this module.